In [1]:
from llama_index.core.bridge.pydantic import BaseModel, Field
from llama_index.llms.ollama import Ollama

In [2]:
class Guess(BaseModel):
    answer: str = Field(description="The answer to the clue, fitting the specified length and any known letters. The answer should be in uppercase and should not contain spaces or punctuation.")
    confidence_score: int = Field(description="A confidence score between 0 and 100 indicating the likelihood that this answer is correct.")
    explanation: str = Field(description="A brief explanation of how the clue leads to this answer.")

class Guesses(BaseModel):
    guesses: list[Guess] = Field(
        description="A list of five potential answers that fit the clue."
    )

In [3]:
llm = Ollama(
    model="llama3.1:latest",
    request_timeout=1200.0,
    context_window=1000,
    temperature=0.1,
    json_mode=True,
)

sllm = llm.as_structured_llm(Guesses)

In [4]:
ordinal_map = {
    1: "first",
    2: "second",
    3: "third",
    4: "fourth",
    5: "fifth",
    6: "sixth",
    7: "seventh",
    8: "eighth",
    9: "ninth",
    10: "tenth",
}


def generate_prompt(clue: str, pattern: list[str]) -> str:
    length = len(pattern)
    pattern_str = "".join(
        [f"The {ordinal_map.get(i, f'{i}th')} letter is {letter if letter != '_' else 'unknown'},\n" for i, letter in enumerate(pattern, start=1)]
    )
    
    return f"""You are a crossword solver. You must give me five guesses that fit the clue and the exact letter pattern.

Constraints:
- Clue: {clue}
- Length: {length} letters

{pattern_str}

Final Output:
Return only the matching words in ALL CAPS, your confidence score (0-100), and an explanation. Each guess should be unique and should not contain spaces or punctuation.
"""

In [5]:
clue = "Black and white animal"
pattern = ["_", "_", "B", "_", "_"]

prompt = generate_prompt(clue, pattern)

print(prompt)

You are a crossword solver. You must give me five guesses that fit the clue and the exact letter pattern.

Constraints:
- Clue: Black and white animal
- Length: 5 letters

The first letter is unknown,
The second letter is unknown,
The third letter is B,
The fourth letter is unknown,
The fifth letter is unknown,


Final Output:
Return only the matching words in ALL CAPS, your confidence score (0-100), and an explanation. Each guess should be unique and should not contain spaces or punctuation.



In [6]:
response = sllm.complete(prompt).raw
response.guesses.sort(key=lambda x: x.confidence_score, reverse=True)

In [7]:
response.guesses

[Guess(answer='ZEBRA', confidence_score=80, explanation='Common black and white animal'),
 Guess(answer='PANDA', confidence_score=60, explanation='Black and white bear species'),
 Guess(answer='DOLPH', confidence_score=40, explanation='Some dolphins have black and white coloring'),
 Guess(answer='DABBY', confidence_score=20, explanation='Uncommon term for a black and white animal'),
 Guess(answer='LEOPA', confidence_score=0, explanation='Not a valid word')]